# Обнаружение ошибок по замороженным активациям B0

Новая задача: предсказать **ошибку B0**, включая уверенные ошибки внутри регионов.
Это не прежняя регрессия пользы B2. **B0 и B2 не обучаются** и не загружаются при обучении головы.

Четыре начальных варианта: уверенность; F2/F3; уверенность + F2/F3; совместная голова с разреженным автоэнкодером.
Сравнение с исходной энтропией выполняется отдельно. OHEM — дополнительная опция, не обязательный первый опыт.

Данные: прежние кэши Cityscapes, 240 train / 60 validation. P1 не запускается автоматически.
Основная метрика — AP обнаружения ошибок, а не mIoU и не MSE пользы.


## 1. Размещение файлов

Если код ещё не установлен, загрузите `frozen-error-head-v1.zip` в корень
`/home/jupyter/project/error-geometry-routing-frozen/` и выполните следующую ячейку.
Она добавляет только новые модули, тест, документацию и этот ноутбук. Старые файлы экспериментов не меняются.
Если код уже установлен, отсутствие ZIP допустимо. Нужен прежний модуль `routing.frozen.common`.


In [2]:
from pathlib import Path
import json
import os
import sys
import time
import subprocess
import zipfile
import numpy as np

project_dir = Path('/home/jupyter/project/error-geometry-routing-frozen')
cache_root = project_dir / 'results/frozen/routing_v1_20261025_132318'
package = project_dir / 'frozen-error-head-v1.zip'
if package.exists():
    with zipfile.ZipFile(package) as z:
        for entry in z.infolist():
            path = Path(entry.filename)
            if entry.is_dir():
                continue
            assert not path.is_absolute() and '..' not in path.parts
            assert (path.as_posix().startswith('routing/error_head/')
                    or path.as_posix().startswith('docs/')
                    or path.as_posix().startswith('notebooks/')
                    or path.as_posix().startswith('tests/')
                    or path.name == 'error_head_package_sha256.json')
            destination = project_dir / path
            content = z.read(entry)
            if destination.exists() and destination.read_bytes() != content:
                if destination.suffix == '.ipynb':
                    continue  # Keep the currently edited notebook and its outputs.
                raise RuntimeError(f'Не перезаписываем изменённый файл: {destination}')
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(content)
assert (project_dir / 'routing/error_head/train.py').exists(), 'Загрузите и распакуйте ZIP'
assert (project_dir / 'routing/frozen/common.py').exists(), 'Нужен код предыдущего frozen-эксперимента'
sys.path.insert(0, str(project_dir))
os.chdir(project_dir)
print('Код:', project_dir / 'routing/error_head')


Код: /home/jupyter/project/error-geometry-routing-frozen/routing/error_head


## 2. Проверка данных и устройства

Эта ячейка не скачивает веса и не запускает B0/B2. Если GPU не выделен, обучение может идти на CPU,
но заметно медленнее. CPU/GPU выбирается явно переменной `device`.


In [3]:
import torch
from routing.error_head.data import checked_split, split_check

train_meta = checked_split(cache_root / 'train', ['F2', 'F3'])
val_meta = checked_split(cache_root / 'val', ['F2', 'F3'])
split_check(train_meta, val_meta)
print('Кадров train / val:', len(train_meta['images']), len(val_meta['images']))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Обучается только новая голова. Сегментаторы не загружаются.')


Кадров train / val: 240 60
Устройство: cuda
GPU: Tesla T4
Обучается только новая голова. Сегментаторы не загружаются.


## 3. Отдельная папка запуска и логи

Первый запуск создаёт отдельную папку и сохраняет путь. Повторное выполнение продолжает работу с ней,
пропуская только завершённые варианты. Для совершенно нового эксперимента задайте другое `session_name`.
Прерывание ячейки останавливает её дочерний процесс. Потеря интернета браузера не равна остановке VM.


In [4]:
session_name = 'error_head_v1_seed0'
experiment_root = cache_root / session_name
experiment_root.mkdir(exist_ok=True)
print('Результаты:', experiment_root)

def run_logged(name, arguments):
    log_path = experiment_root / f'{name}.log'
    env = os.environ.copy()
    for key in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
        env[key] = '4'
    started = time.perf_counter()
    print(f'\n=== {name} ===', flush=True)
    with log_path.open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', *map(str, arguments)],
                                   cwd=project_dir, env=env, stdout=log, stderr=subprocess.STDOUT)
        last = started
        try:
            while process.poll() is None:
                time.sleep(2)
                now = time.perf_counter()
                if now - last >= 30:
                    tail = log_path.read_text(errors='replace').splitlines()[-3:]
                    print(f'{name}: {(now-started)/60:.1f} мин; ' + ' | '.join(tail), flush=True)
                    last = now
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    print(log_path.read_text(errors='replace')[-8000:])
    print(f'{name}: код {process.returncode}, {(time.perf_counter()-started)/60:.1f} мин')
    if process.returncode:
        raise RuntimeError(f'Ошибка. Лог: {log_path}')


Результаты: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/error_head_v1_seed0


## 4. Тесты и нормализация на train

Нормализация нужна один раз для всех вариантов F2/F3. Validation в её расчёте не участвует.


In [5]:
check = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                        '-p', 'test_error_head.py', '-v'], cwd=project_dir,
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(check.stdout)
assert check.returncode == 0

normalization_dir = experiment_root / 'normalization'
if normalization_dir.exists():
    from routing.error_head.data import file_hash
    nm = json.loads((normalization_dir / 'config.json').read_text())
    assert set(nm['train_manifest']['images']) == set(train_meta['images'])
    assert nm['sha256'] == file_hash(normalization_dir / 'normalization.npz')
    print('Нормализация уже готова')
else:
    run_logged('normalization', ['routing.error_head.prepare', '--train', cache_root / 'train',
                                '--out', normalization_dir, '--layers', 'F2,F3'])


test_histogram_perfect_reversed_and_ties (test_error_head.ErrorCPU) ... ok
test_interior_excludes_predicted_boundary (test_error_head.ErrorCPU) ... ok
test_split_guard_and_bad_ignore (test_error_head.ErrorCPU) ... ok
test_complete_cli_train_eval_and_frozen_thresholds (test_error_head.ErrorTorch) ... ok
test_detachment_and_sparse_ae_gradients (test_error_head.ErrorTorch) ... ok
test_pooled_error_fraction_and_ignored_loss (test_error_head.ErrorTorch) ... ok

----------------------------------------------------------------------
Ran 6 tests in 20.161s

OK


=== normalization ===
normalization 40/240
normalization 80/240
normalization 120/240
normalization 160/240
normalization 200/240
normalization 240/240
Normalization saved: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/error_head_v1_seed0/normalization

normalization: код 0, 0.2 мин


## 5. Обучение только новых голов

Максимум 8 эпох, ранняя остановка после 3 эпох без улучшения обычной validation BCE.
Это первый скрининг с seed=0. Время заранее не фиксируем: первый вариант покажет скорость на вашем ресурсе.

SAE обучается совместно: $L=L_{BCE}+0.1L_{reconstruction}+0.001L_{sparsity}$.
Он сжимает стандартизованные активации в 16 неотрицательных каналов. Его восстановление само по себе
не доказывает обнаружение ошибок. На этом этапе мы не подбираем Махаланобис и kNN в новом пространстве.


In [6]:
jobs = {
    'confidence_bce': ['--mode', 'confidence'],
    'activations_bce': ['--mode', 'activations'],
    'both_bce': ['--mode', 'both'],
    'both_sparse_ae': ['--mode', 'both', '--sparse-ae'],
}
# Дополнительный контролируемый опыт, если понадобится позже:
# jobs['both_ohem'] = ['--mode', 'both', '--hard-fraction', '0.25']
# Для проверки влияния числа параметров у финалиста:
# jobs['confidence_w32'] = ['--mode', 'confidence', '--width', '32']

for name, extra in jobs.items():
    out_dir = experiment_root / name
    if out_dir.exists():
        config_path = out_dir / 'config.json'
        cfg = json.loads(config_path.read_text()) if config_path.exists() else {}
        if cfg.get('complete') and (out_dir / 'head.pt').exists():
            assert cfg['seed'] == 0
            assert set(cfg['train_manifest']['images']) == set(train_meta['images'])
            assert set(cfg['val_manifest']['images']) == set(val_meta['images'])
            print(name, 'уже обучен — пропускаем')
            continue
        raise RuntimeError(f'Незавершённый вариант: {out_dir}. Сохраните его отдельно перед повтором.')
    run_logged('train_' + name, [
        'routing.error_head.train', '--train', cache_root / 'train', '--val', cache_root / 'val',
        '--out', out_dir, '--normalization', normalization_dir,
        '--layers', 'F2,F3', '--width', '16', '--latent', '16',
        '--epochs', '8', '--patience', '3', '--seed', '0', '--device', device, *extra])



=== train_confidence_bce ===
train_confidence_bce: 0.5 мин; epoch 2 train 160/240 | epoch 2 train 200/240 | epoch 2 train 240/240
train_confidence_bce: 1.0 мин; epoch 4 train 160/240 | epoch 4 train 200/240 | epoch 4 train 240/240
train_confidence_bce: 1.5 мин; epoch 6 train 160/240 | epoch 6 train 200/240 | epoch 6 train 240/240
train_confidence_bce: 2.0 мин; epoch 8 train 200/240 | epoch 8 train 240/240 | epoch 8 val 40/60
{'device': 'cuda', 'trainable_parameters': 2401, 'mode': 'confidence', 'sparse_ae': False, 'note': 'No segmentation model exists in this process; only the head is trained'}
epoch 1 train 40/240
epoch 1 train 80/240
epoch 1 train 120/240
epoch 1 train 160/240
epoch 1 train 200/240
epoch 1 train 240/240
epoch 1 val 40/60
epoch 1 val 60/60
{'epoch': 1, 'train_bce': 0.2396827476454738, 'train_objective': 0.2396827476454738, 'train_reconstruction': 0.0, 'train_latent_l1': 0.0, 'train_latent_zero_fraction': 0.0, 'val_bce': 0.12395561648403576, 'val_objective': 0.1239556

## 6. Validation: обнаружение ошибок и отдельно кэшированный роутинг

Оцениваем все валидные пиксели, внутренние области компонент, низкоэнтропийные пиксели и их пересечение.
Уверенность здесь заранее определена как $H/\log(19)\le0.1$; внутренность — дальше 3 пикселей от границ B0.
Кандидаты не фильтруются энтропией. AP приближённая: 4096 бинов; все доли в диапазоне 0–1.

Опция `--routing` использует уже сохранённые ответы B2, не запускает B2 на GPU и не измеряет сквозное время.
Порог на validation подбирается под 0.8 и 2.4 окна/кадр. Из-за совпадающих оценок точное число окон может отличаться.


In [7]:
methods = {'raw_entropy': None, **{name: experiment_root / name for name in jobs}}
for name, run in methods.items():
    report_path = experiment_root / f'{name}_validation.json'
    if report_path.exists():
        saved = json.loads(report_path.read_text())
        assert set(saved['cache_manifest']['images']) == set(val_meta['images'])
        print(name, 'validation уже готова')
        continue
    extra = ['--run', run] if run else ['--score', 'entropy']
    run_logged('validate_' + name, [
        'routing.error_head.evaluate', '--cache', cache_root / 'val', '--out', report_path,
        '--device', device, '--routing', '--bins', '4096', '--interior-radius', '3',
        '--low-entropy', '0.1', '--target-windows', '0.8,2.4', *extra])



=== validate_raw_entropy ===
evaluate 20/60
evaluate 40/60
evaluate 60/60
{'metrics': {'all': {'pixels': 28017487, 'errors': 1268244, 'error_prevalence': 0.04526615823895983, 'average_precision_binned': 0.4344521531619697, 'recall_at_selected_fraction': {'0.01': 0.12687703258758007, '0.05': 0.4908510335009728, '0.1': 0.7707574140376959, '0.2': 0.9473538904974121}}, 'interior': {'pixels': 24938062, 'errors': 420619, 'error_prevalence': 0.01686654720803886, 'average_precision_binned': 0.2661736133575184, 'recall_at_selected_fraction': {'0.01': 0.21724237103910515, '0.05': 0.6346424100793532, '0.1': 0.8269045699055836, '0.2': 0.9359939205247795}}, 'low_entropy': {'pixels': 24561383, 'errors': 203096, 'error_prevalence': 0.008268915475972994, 'average_precision_binned': 0.07313635589218721, 'recall_at_selected_fraction': {'0.01': 0.12307892388177653, '0.05': 0.4836629311077988, '0.1': 0.7133852354442125, '0.2': 0.8813735213269448}}, 'low_entropy_interior': {'pixels': 23842301, 'errors': 1

## 7. Таблицы

Сначала сравните `both_bce` и `activations_bce` с обученным `confidence_bce`.
Проверьте AP всех ошибок и отдельно уверенных внутренних ошибок. AP нельзя интерпретировать как mIoU.
Более высокая AP ещё не означает, что B2 исправляет найденные ошибки.


In [8]:
import csv
rows = []
routing_rows = []
for name in methods:
    report = json.loads((experiment_root / f'{name}_validation.json').read_text())
    metrics = report['metrics']
    cfg = json.loads((experiment_root / name / 'config.json').read_text()) if name != 'raw_entropy' else {}
    rows.append({
        'method': name,
        'best_epoch': cfg.get('best_epoch'),
        'parameters': cfg.get('trainable_parameters', 0),
        'AP_all': metrics['all']['average_precision_binned'],
        'recall_top10pct': metrics['all']['recall_at_selected_fraction'].get('0.1'),
        'AP_low_entropy_interior': metrics['low_entropy_interior']['average_precision_binned'],
        'low_entropy_interior_errors': metrics['low_entropy_interior']['errors'],
        'AP_components': metrics['components']['average_precision_binned'],
        'Brier': report['pixel_brier'],
    })
    for point in report['routing_rows']:
        routing_rows.append({'method': name, 'validation_target_windows': point['target_validation_windows'],
                             'actual_windows': point['mean_windows'], 'cached_mIoU': point['cached_miou']})

def print_save_table(name, records):
    path = experiment_root / name
    with path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
    try:
        import pandas as pd
        print(pd.DataFrame(records).round(5).to_string(index=False))
    except ImportError:
        for row in records:
            print(row)
    print('Сохранено:', path)

print_save_table('validation_error_detection.csv', rows)
print_save_table('validation_cached_routing.csv', routing_rows)
print('Нет сквозных замеров времени. P1 автоматически не запускалась.')


         method  best_epoch  parameters  AP_all  recall_top10pct  AP_low_entropy_interior  low_entropy_interior_errors  AP_components   Brier
    raw_entropy         NaN           0 0.43445          0.77076                  0.07173                       169927        0.77155     NaN
 confidence_bce         8.0        2401 0.42945          0.75445                  0.06261                       169927        0.74831 0.03135
activations_bce         4.0        6481 0.27549          0.56419                  0.04795                       169927        0.70877 0.03661
       both_bce         8.0        6529 0.41091          0.74352                  0.06055                       169927        0.74047 0.03178
 both_sparse_ae         6.0       10081 0.42800          0.75386                  0.06730                       169927        0.74514 0.03133
Сохранено: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/error_head_v1_seed0/validation_error_detect

## 8. Решение после validation

Если активации не добавили сигнала к обученному контролю, сохраните отрицательный результат.
Если добавили — сначала повторите финалиста и контроль с seeds 1 и 2 в отдельных папках,
проверьте влияние числа параметров, затем оцените на новых размеченных изображениях.
Историческая P1 уже исследовалась и имеет пересечение последовательностей с train;
называть её новым независимым тестом нельзя. Сквозной GPU-замер нужен только для финалиста.

Не включаем автоматически: изменение весов B0/B2, инъекцию токенов, подавление активаций основной модели,
новую геометрию, поиск гиперпараметров на P1.


## 9. Сохранить результаты

Архив включает новые головы, нормализацию, конфигурации, логи, таблицы и код.
Большие покадровые кэши и изображения не копируются. После завершения всех процессов и сохранения
ноутбука GPU-ВМ можно освободить отдельно в интерфейсе DataSphere.


In [9]:
import shutil
export_dir = project_dir / 'exports'
export_dir.mkdir(exist_ok=True)
archive = export_dir / (session_name + '_results.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for root in [experiment_root, project_dir / 'routing/error_head']:
        for path in root.rglob('*'):
            if path.is_file() and '__pycache__' not in path.parts:
                z.write(path, path.relative_to(project_dir))
    for relative in ['tests/test_error_head.py', 'docs/frozen_error_head_experiment.md',
                     'notebooks/frozen_error_head_v1.ipynb']:
        path = project_dir / relative
        if path.exists():
            z.write(path, relative)
    for split in ['train', 'val', 'test']:
        path = cache_root / split / 'manifest.json'
        if path.exists():
            z.write(path, path.relative_to(project_dir))
print('Скачать:', archive)
print('Размер MiB:', round(archive.stat().st_size / 2**20, 2))


Скачать: /home/jupyter/project/error-geometry-routing-frozen/exports/error_head_v1_seed0_results.zip
Размер MiB: 0.39
